# Step 1: Data Loading & Storage

Load Lending Club CSV data and convert to Parquet format using PySpark.

In [1]:
import socket, sys
print("HOSTNAME:", socket.gethostname())
print("PYTHON:", sys.executable)

HOSTNAME: 95885c191229
PYTHON: /opt/conda/bin/python


In [2]:
!which python
!python -c "import pyspark; print(pyspark.__file__)"

/opt/conda/bin/python
/opt/conda/lib/python3.11/site-packages/pyspark/__init__.py


In [3]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

from spark_utils import get_spark_session, csv_to_parquet, load_parquet

In [4]:
# Toggle: True = 1% sample for fast dev, False = full data
SAMPLE_MODE = False

## 1.1 Initialize Spark Session

In [5]:
spark = get_spark_session()

Spark driver memory: 7g


## 1.2 Load Accepted Loans CSV

In [6]:
df_accepted = spark.read.csv(
    '/home/jovyan/work/data/raw/accepted_2007_to_2018Q4.csv',
    header=True, inferSchema=True
)
print(f'Rows: {df_accepted.count()}, Columns: {len(df_accepted.columns)}')

Rows: 2260701, Columns: 151


In [7]:
df_accepted.printSchema()

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string 

## 1.3 Load Rejected Loans CSV

In [8]:
df_rejected = spark.read.csv(
    '/home/jovyan/work/data/raw/rejected_2007_to_2018Q4.csv',
    header=True, inferSchema=True
)
print(f'Rows: {df_rejected.count()}, Columns: {len(df_rejected.columns)}')

Rows: 27648741, Columns: 9


## 1.4 Sample (if SAMPLE_MODE)

In [9]:
if SAMPLE_MODE:
    df_accepted = df_accepted.sample(fraction=0.01, seed=42)
    df_rejected = df_rejected.sample(fraction=0.01, seed=42)
    print(f'Sampled accepted: {df_accepted.count()} rows')
    print(f'Sampled rejected: {df_rejected.count()} rows')

## 1.5 Convert to Parquet

In [10]:
df_accepted.write.mode('overwrite').parquet('/home/jovyan/work/data/parquet/accepted_raw.parquet')
df_rejected.write.mode('overwrite').parquet('/home/jovyan/work/data/parquet/rejected_raw.parquet')
print('Parquet files saved.')

Parquet files saved.


## 1.6 Verify Parquet

In [11]:
df_check = spark.read.parquet('/home/jovyan/work/data/parquet/accepted_raw.parquet')
print(f'Parquet rows: {df_check.count()}')
df_check.show(5)

Parquet rows: 2260701
+-------+---------+---------+-----------+---------------+----------+--------+-----------+-----+---------+--------------------+----------+--------------+----------+-------------------+--------+-----------+----------+--------------------+--------------------+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+---------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+------------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------+-------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+----------